In [ ]:
import numpy as np
import pickle
import scipy
def get_llr(t, mu_in, mu_out, sigma_in, sigma_out, eps = 1e-12):
    sigma_in  = np.maximum(sigma_in,  eps)
    sigma_out = np.maximum(sigma_out, eps)
    return np.log(sigma_out) - np.log(sigma_in) + 0.5 * (t - mu_out)**2/ sigma_out**2 - 0.5 * (t - mu_in)**2/ sigma_in**2


In [ ]:
with open("results/adult/in_indices_target.pkl","rb") as f:
    in_indices = pickle.load(f)
with open("results/adult/stats_target.pkl","rb") as f:
    stats = pickle.load(f)

In [ ]:
M, N = stats.shape
assert in_indices.shape == (M, N), "in_indices must be (M, N)"
ALPHA = 0.01
Ms = [2**i for i in range(4,14)]
M_px_min = int(1/ALPHA)
in_mask  = in_indices.astype(bool)
out_mask = ~in_mask

# ------------------------------------------------------------------
# Element-wise masks
# ------------------------------------------------------------------
stats_in  = np.where(in_mask,  stats, 0.0)   # (M, N)
stats_out = np.where(out_mask, stats, 0.0)   # (M, N)
stats_in_sq  = np.where(in_mask,  stats ** 2, 0.0)
stats_out_sq = np.where(out_mask, stats ** 2, 0.0)

col_sum_in    = stats_in.sum(axis=0)      # (N,)
col_sum_out   = stats_out.sum(axis=0)
col_sum_in_sq = stats_in_sq.sum(axis=0)
col_sum_out_sq= stats_out_sq.sum(axis=0)
col_cnt_in    = in_mask.sum(axis=0)       # (N,)  int
col_cnt_out   = out_mask.sum(axis=0)

# ------------------------------------------------------------------
# LOO sums / counts  (subtract the m-th model's contribution)
# Shapes broadcast to (M, N) automatically
# ------------------------------------------------------------------
loo_cnt_in  = col_cnt_in  - in_mask.astype(int)   # (M, N)
loo_cnt_out = col_cnt_out - out_mask.astype(int)

loo_sum_in     = col_sum_in    - stats_in          # (M, N)
loo_sum_out    = col_sum_out   - stats_out
loo_sum_in_sq  = col_sum_in_sq - stats_in_sq
loo_sum_out_sq = col_sum_out_sq- stats_out_sq

mu_ins  = loo_sum_in  / loo_cnt_in   # (M, N)
mu_outs = loo_sum_out / loo_cnt_out

sigma_ins  = np.sqrt(np.maximum(loo_sum_in_sq  / loo_cnt_in  - mu_ins  ** 2, 0.0))
sigma_outs = np.sqrt(np.maximum(loo_sum_out_sq / loo_cnt_out - mu_outs ** 2, 0.0))

# --------------------------
# Computing LLR grid
# --------------------------
llr_grid = get_llr(stats, mu_ins, mu_outs, sigma_ins, sigma_outs)

# --------------------------
# Computing PP Stats Grid
# --------------------------
delta_mu = mu_ins - mu_outs
stats_pp = (np.sign(delta_mu) * (stats - mu_outs)) / sigma_outs

In [ ]:
TAUS_ANALYTICAL, TAUS_ALL, TAUS_ALL_PP, TAUS_PP_TDIST = np.zeros(len(Ms)), np.zeros(len(Ms)), np.zeros(len(Ms)), np.zeros(len(Ms))
for i in range(len(Ms)):
    M_i = Ms[i]
    curr_stats_pp, curr_in_indices, curr_llrs = stats_pp[:M_i,:], in_indices[:M_i,:], llr_grid[:M_i,:]
    mask_out = ~curr_in_indices

    out_stats_pp = curr_stats_pp[mask_out]
    # loc = np.mean(out_stats_pp)
    # scale = np.std(out_stats_pp, ddof=0)

    # print(loc,scale)

    tau_analytical = scipy.stats.norm.isf(ALPHA, loc=0.0, scale=1.0)

    tau_all = np.nanquantile(np.where(mask_out, curr_llrs, np.nan), 1 - ALPHA)

    tau_all_pp = np.nanquantile(np.where(mask_out, curr_stats_pp, np.nan), 1 - ALPHA)

    df, _, _ = scipy.stats.t.fit(out_stats_pp, floc=0.0, fscale=1.)
    tau_pp_tdist = scipy.stats.t.isf(ALPHA, df=df, loc=0.0, scale=1.) 

    TAUS_ANALYTICAL[i] = tau_analytical
    TAUS_ALL[i] = tau_all
    TAUS_ALL_PP[i] = tau_all_pp
    TAUS_PP_TDIST[i] = tau_pp_tdist


In [ ]:
TPR_concat, TPR_concat_pp = np.zeros(len(Ms)), np.zeros(len(Ms))
TPR_concat_pp_tdist, TPR_concat_pp_norm = np.zeros(len(Ms)),  np.zeros(len(Ms))
TPR_px, TPR_px_ana = [],[]
Ms_PX = []
for i in range(len(Ms)):
    M_i = Ms[i]
    curr_stats_pp, curr_in_indices, curr_llrs = stats_pp[:M_i,:], in_indices[:M_i,:], llr_grid[:M_i,:]
    
    mask_out = ~curr_in_indices
    mask_in  = curr_in_indices

    # tau_all = np.nanquantile(np.where(mask_out, curr_llrs, np.nan), 1 - ALPHA)
    tpr_all = ((curr_llrs > TAUS_ALL[i]) & mask_in).sum() / mask_in.sum()
    TPR_concat[i] = tpr_all

    # tau_all_pp = np.nanquantile(np.where(mask_out, curr_stats_pp, np.nan), 1 - ALPHA)
    tpr_all_pp = ((curr_stats_pp > TAUS_ALL_PP[i]) & mask_in).sum() / mask_in.sum()
    TPR_concat_pp[i] = tpr_all_pp

    tpr_all_norm = ((curr_stats_pp > TAUS_ANALYTICAL[i]) & mask_in).sum() / mask_in.sum()
    TPR_concat_pp_norm[i] = tpr_all_norm

    if M_i >= M_px_min:
        tau_px = np.nanquantile(np.where(mask_out, curr_llrs, np.nan), 1 - ALPHA, axis=0)
        tpr_px = (((curr_llrs > tau_px) & mask_in).sum(axis=0)
                    / mask_in.sum(axis=0))
        TPR_px.append(tpr_px.mean())

        tpr_px_ana = (((curr_stats_pp > TAUS_ANALYTICAL[i]) & mask_in).sum(axis=0)
                    / mask_in.sum(axis=0))
        TPR_px_ana.append(tpr_px_ana.mean())
        Ms_PX.append(M_i)

    TPR_concat_pp_tdist[i] = (
        ((curr_stats_pp > TAUS_PP_TDIST[i]) & mask_in).sum() / mask_in.sum()
    )

In [ ]:
with open(f"results/adult/tabpfn_N_{N}_M_{M-1}_fpr_{ALPHA}.pkl","wb") as f:
    pickle.dump({
        "LLRS": llr_grid,
        "STATS_PP": stats_pp,
        "IN_INDICES": in_indices,
        "Ms": Ms,
        "TPR_concat": TPR_concat,
        "TPR_concat_pp": TPR_concat_pp,
        "TPR_concat_pp_tdist": TPR_concat_pp_tdist,
        "TPR_concat_pp_norm": TPR_concat_pp_norm,
        "TPR_px": TPR_px,
        "TPR_px_ana": TPR_px_ana,
        "TAUS_ANALYTICAL": TAUS_ANALYTICAL,
        "TAUS_ALL_PP": TAUS_ALL_PP,
        "TAUS_PP_TDIST": TAUS_PP_TDIST,
        "TAUS_ALL": TAUS_ALL,
        "Ms_PX": Ms_PX
        
    }, f)

### Non-Oracle Approach:

In [ ]:
EPS = 1e-12

def safe_divide(num, den, fill=np.nan):
    """
    Elementwise num / den, but returns `fill` wherever den == 0.
    """
    num = np.asarray(num, dtype=float)
    den = np.asarray(den)
    return np.divide(
        num,
        den,
        out=np.full_like(num, fill, dtype=float),
        where=(den != 0)
    )

all_stats = np.copy(stats).astype(float, copy=False)
all_in_indices = np.copy(in_indices)

LLRs = {}
STATS_PP = {}

for m in range(len(Ms)):
    M_total = Ms[m]

    stats = all_stats[:M_total + 1, :]
    in_indices = all_in_indices[:M_total + 1, :]

    M, N = stats.shape
    assert in_indices.shape == (M, N), "in_indices must be (M, N)"

    in_mask = in_indices.astype(bool)
    out_mask = ~in_mask

    # ------------------------------------------------------------------
    # Element-wise masks
    # ------------------------------------------------------------------
    stats_sq = stats ** 2

    stats_in = np.where(in_mask, stats, 0.0)
    stats_out = np.where(out_mask, stats, 0.0)

    stats_in_sq = np.where(in_mask, stats_sq, 0.0)
    stats_out_sq = np.where(out_mask, stats_sq, 0.0)

    col_sum_in = stats_in.sum(axis=0)
    col_sum_out = stats_out.sum(axis=0)

    col_sum_in_sq = stats_in_sq.sum(axis=0)
    col_sum_out_sq = stats_out_sq.sum(axis=0)

    col_cnt_in = in_mask.sum(axis=0)
    col_cnt_out = out_mask.sum(axis=0)

    # ------------------------------------------------------------------
    # LOO sums / counts
    # ------------------------------------------------------------------
    loo_cnt_in = col_cnt_in - in_mask.astype(int)
    loo_cnt_out = col_cnt_out - out_mask.astype(int)

    loo_sum_in = col_sum_in - stats_in
    loo_sum_out = col_sum_out - stats_out

    loo_sum_in_sq = col_sum_in_sq - stats_in_sq
    loo_sum_out_sq = col_sum_out_sq - stats_out_sq

    # ------------------------------------------------------------------
    # Divide-by-zero-safe LOO means
    # Empty LOO groups get nan.
    # ------------------------------------------------------------------
    mu_ins = safe_divide(loo_sum_in, loo_cnt_in)
    mu_outs = safe_divide(loo_sum_out, loo_cnt_out)

    # ------------------------------------------------------------------
    # Divide-by-zero-safe second moments and variances
    # ------------------------------------------------------------------
    ex2_ins = safe_divide(loo_sum_in_sq, loo_cnt_in)
    ex2_outs = safe_divide(loo_sum_out_sq, loo_cnt_out)

    var_ins = ex2_ins - mu_ins ** 2
    var_outs = ex2_outs - mu_outs ** 2

    # Numerical cleanup:
    # - tiny negative values from floating-point cancellation go to 0
    # - truly invalid entries stay nan
    var_ins = np.where(loo_cnt_in > 0, np.maximum(var_ins, 0.0), np.nan)
    var_outs = np.where(loo_cnt_out > 0, np.maximum(var_outs, 0.0), np.nan)

    sigma_ins = np.sqrt(var_ins)
    sigma_outs = np.sqrt(var_outs)

    # ------------------------------------------------------------------
    # Safe sigmas for likelihood calculations
    # If count is valid but variance is zero, use EPS instead of 0.
    # If count is invalid, keep nan.
    # ------------------------------------------------------------------
    sigma_ins_safe = np.where(
        loo_cnt_in > 0,
        np.maximum(sigma_ins, EPS),
        np.nan
    )

    sigma_outs_safe = np.where(
        loo_cnt_out > 0,
        np.maximum(sigma_outs, EPS),
        np.nan
    )

    valid_in = (
        (loo_cnt_in > 0)
        & np.isfinite(mu_ins)
        & np.isfinite(sigma_ins_safe)
    )

    valid_out = (
        (loo_cnt_out > 0)
        & np.isfinite(mu_outs)
        & np.isfinite(sigma_outs_safe)
    )

    valid_llr = valid_in & valid_out

    # ------------------------------------------------------------------
    # Computing LLR grid
    # ------------------------------------------------------------------
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        llr_grid = get_llr(
            stats,
            mu_ins,
            mu_outs,
            sigma_ins_safe,
            sigma_outs_safe
        )

    # Mark entries with insufficient LOO data as nan.
    llr_grid = np.where(valid_llr, llr_grid, np.nan)

    LLRs[M_total] = llr_grid

    # ------------------------------------------------------------------
    # Computing PP Stats Grid
    # ------------------------------------------------------------------
    delta_mu = mu_ins - mu_outs

    numerator = np.sign(delta_mu) * (stats - mu_outs)

    valid_pp = (
        valid_out
        & np.isfinite(delta_mu)
        & np.isfinite(numerator)
    )

    stats_pp = np.divide(
        numerator,
        sigma_outs_safe,
        out=np.full_like(stats, np.nan, dtype=float),
        where=valid_pp
    )

    STATS_PP[M_total] = stats_pp

In [ ]:
TAUS_ANALYTICAL, TAUS_ALL, TAUS_ALL_PP = np.zeros(len(Ms)), np.zeros(len(Ms)), np.zeros(len(Ms))
TAUS_PP_TDIST, Ms_TDIST = [], []
for i in range(len(Ms)):
    M_i = Ms[i]
    curr_stats_pp, curr_in_indices, curr_llrs = STATS_PP[M_i], all_in_indices[:M_i+1,:], LLRs[M_i]
    mask_out = ~curr_in_indices

    out_stats_pp = curr_stats_pp[mask_out]

    tau_analytical = scipy.stats.norm.isf(ALPHA, loc=0.0, scale=1.0)
    tau_all = np.nanquantile(np.where(mask_out, curr_llrs, np.nan), 1 - ALPHA)
    tau_all_pp = np.nanquantile(np.where(mask_out, curr_stats_pp, np.nan), 1 - ALPHA)
    
    if np.isfinite(out_stats_pp).all():
        df, _, _ = scipy.stats.t.fit(out_stats_pp, floc=0.0, fscale=1.)
        tau_pp_tdist = scipy.stats.t.isf(ALPHA, df=df, loc=0.0, scale=1.) 
        TAUS_PP_TDIST.append(tau_pp_tdist)
        Ms_TDIST.append(M_i)

    TAUS_ANALYTICAL[i] = tau_analytical
    TAUS_ALL[i] = tau_all
    TAUS_ALL_PP[i] = tau_all_pp

In [ ]:
TPR_concat, TPR_concat_pp = np.zeros(len(Ms)), np.zeros(len(Ms))
TPR_concat_pp_norm = np.zeros(len(Ms))
TPR_px, TPR_px_ana = [],[]
Ms_PX = []
for i in range(len(Ms)):
    M_i = Ms[i]
    curr_stats_pp, curr_in_indices, curr_llrs = STATS_PP[M_i], all_in_indices[:M_i+1,:], LLRs[M_i]
    
    mask_out = ~curr_in_indices
    mask_in  = curr_in_indices

    tpr_all = ((curr_llrs > TAUS_ALL[i]) & mask_in).sum() / mask_in.sum()
    TPR_concat[i] = tpr_all

    tpr_all_pp = ((curr_stats_pp > TAUS_ALL_PP[i]) & mask_in).sum() / mask_in.sum()
    TPR_concat_pp[i] = tpr_all_pp

    tpr_all_norm = ((curr_stats_pp > TAUS_ANALYTICAL[i]) & mask_in).sum() / mask_in.sum()
    TPR_concat_pp_norm[i] = tpr_all_norm

    if M_i >= M_px_min:
        tau_px = np.nanquantile(np.where(mask_out, curr_llrs, np.nan), 1 - ALPHA, axis=0)
        tpr_px = (((curr_llrs > tau_px) & mask_in).sum(axis=0)
                    / mask_in.sum(axis=0))
        TPR_px.append(tpr_px.mean())

        tpr_px_ana = (((curr_stats_pp > TAUS_ANALYTICAL[i]) & mask_in).sum(axis=0)
                    / mask_in.sum(axis=0))
        TPR_px_ana.append(tpr_px_ana.mean())
        Ms_PX.append(M_i)

TPR_concat_pp_tdist = []
for i in range(len(Ms_TDIST)):
    TPR_concat_pp_tdist.append((
            ((curr_stats_pp > TAUS_PP_TDIST[i]) & mask_in).sum() / mask_in.sum()
        ))

In [ ]:
with open(f"results/adult/tabpfn_N_{N}_M_{M-1}_fpr_{ALPHA}_NOr.pkl","wb") as f:
    pickle.dump({
        "LLRS": llr_grid,
        "STATS_PP": stats_pp,
        "IN_INDICES": all_in_indices,
        "Ms": Ms,
        "TPR_concat": TPR_concat,
        "TPR_concat_pp": TPR_concat_pp,
        "TPR_concat_pp_tdist": TPR_concat_pp_tdist,
        "TPR_concat_pp_norm": TPR_concat_pp_norm,
        "TPR_px": TPR_px,
        "TPR_px_ana": TPR_px_ana,
        "TAUS_ANALYTICAL": TAUS_ANALYTICAL,
        "TAUS_ALL_PP": TAUS_ALL_PP,
        "TAUS_PP_TDIST": TAUS_PP_TDIST,
        "TAUS_ALL": TAUS_ALL,
        "Ms_TDIST": Ms_TDIST,
        "Ms_PX": Ms_PX
        
    }, f)